In [1]:
# ===================================
# Useful Imports: Add more as needed
# ===================================

# Standard Libraries
import os
import time
import math
import io
import zipfile
import requests
from urllib.parse import urlparse
from itertools import chain, combinations

# Data Science Libraries
import numpy as np
import pandas as pd
import seaborn as sns

# Visualization
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.ticker as mticker  # Optional: Format y-axis labels as dollars
import seaborn as sns

# Scikit-learn (Machine Learning)
from sklearn.model_selection import (
    train_test_split, 
    cross_val_score, 
    GridSearchCV, 
    RandomizedSearchCV, 
    RepeatedKFold
)
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error
from sklearn.feature_selection import SequentialFeatureSelector, f_regression, SelectKBest
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor, GradientBoostingRegressor

# Progress Tracking

from tqdm import tqdm

# =============================
# Global Variables
# =============================
random_state = 42

# =============================
# Utility Functions
# =============================

# Format y-axis labels as dollars with commas (optional)
def dollar_format(x, pos):
    return f'${x:,.0f}'

# Convert seconds to HH:MM:SS format
def format_hms(seconds):
    return time.strftime("%H:%M:%S", time.gmtime(seconds))

In [2]:
# feature selection per model
best_features_per_model = {'Linear Regression': ['calculatedfinishedsquarefeet',
  'prod_bedroomcnt_bedroomcnt',
  'prod_bathroomcnt_bathroomcnt',
  'age',
  'buildingqualitytypeid',
  'heatingorsystemtypeid_7.0',
  'log_age',
  'roomcnt',
  'fips_6059.0',
  'prod_fireplacecnt_garagecarcnt',
  'airconditioningtypeid_13.0',
  'threequarterbathnbr',
  'garagetotalsqft',
  'lotsizesquarefeet',
  'prod_numberofstories_numberofstories',
  'heatingorsystemtypeid_24.0',
  'airconditioningtypeid_11.0',
  'airconditioningtypeid_9.0',
  'heatingorsystemtypeid_18.0',
  'heatingorsystemtypeid_11.0'],
 'Ridge Regression': ['calculatedfinishedsquarefeet',
  'prod_bedroomcnt_bedroomcnt',
  'prod_bathroomcnt_bathroomcnt',
  'age',
  'buildingqualitytypeid',
  'heatingorsystemtypeid_7.0',
  'log_age',
  'roomcnt',
  'fips_6059.0',
  'prod_fireplacecnt_garagecarcnt',
  'airconditioningtypeid_13.0',
  'threequarterbathnbr',
  'garagetotalsqft',
  'lotsizesquarefeet',
  'prod_numberofstories_numberofstories',
  'heatingorsystemtypeid_24.0',
  'airconditioningtypeid_11.0',
  'airconditioningtypeid_9.0',
  'heatingorsystemtypeid_18.0',
  'heatingorsystemtypeid_11.0',
  'regionidcounty_1286.0'],
 'Lasso Regression': ['calculatedfinishedsquarefeet',
  'prod_bedroomcnt_bedroomcnt',
  'prod_bathroomcnt_bathroomcnt',
  'age',
  'buildingqualitytypeid',
  'heatingorsystemtypeid_7.0',
  'log_age',
  'roomcnt',
  'fips_6059.0',
  'prod_fireplacecnt_garagecarcnt',
  'airconditioningtypeid_13.0',
  'threequarterbathnbr',
  'garagetotalsqft',
  'lotsizesquarefeet',
  'prod_numberofstories_numberofstories',
  'heatingorsystemtypeid_24.0',
  'airconditioningtypeid_11.0',
  'airconditioningtypeid_9.0',
  'heatingorsystemtypeid_18.0',
  'heatingorsystemtypeid_11.0',
  'regionidcounty_1286.0'],
 'Decision Tree': ['prod_bathroomcnt_bathroomcnt',
  'buildingqualitytypeid',
  'unitcnt',
  'regionidcounty_3101.0',
  'roomcnt',
  'fips_6037.0',
  'heatingorsystemtypeid_6.0',
  'heatingorsystemtypeid_24.0',
  'heatingorsystemtypeid_18.0',
  'heatingorsystemtypeid_13.0',
  'heatingorsystemtypeid_1.0'],
 'Bagging': ['prod_bathroomcnt_bathroomcnt',
  'buildingqualitytypeid',
  'unitcnt',
  'heatingorsystemtypeid_2.0',
  'regionidcounty_3101.0',
  'heatingorsystemtypeid_24.0',
  'fips_6111.0',
  'airconditioningtypeid_5.0',
  'prod_fireplacecnt_garagecarcnt',
  'threequarterbathnbr',
  'airconditioningtypeid_11.0',
  'airconditioningtypeid_9.0',
  'airconditioningtypeid_13.0',
  'heatingorsystemtypeid_18.0',
  'heatingorsystemtypeid_1.0',
  'heatingorsystemtypeid_6.0',
  'fips_6037.0',
  'heatingorsystemtypeid_13.0',
  'airconditioningtypeid_1.0',
  'prod_numberofstories_numberofstories',
  'heatingorsystemtypeid_10.0',
  'heatingorsystemtypeid_7.0',
  'fips_6059.0',
  'regionidcounty_1286.0',
  'heatingorsystemtypeid_11.0',
  'heatingorsystemtypeid_20.0',
  'regionidcounty_2061.0',
  'roomcnt',
  'calculatedfinishedsquarefeet',
  'age',
  'lotsizesquarefeet',
  'prod_bedroomcnt_bedroomcnt'],
 'Random Forest': ['prod_bathroomcnt_bathroomcnt',
  'buildingqualitytypeid',
  'unitcnt',
  'regionidcounty_3101.0',
  'heatingorsystemtypeid_6.0',
  'fips_6059.0',
  'heatingorsystemtypeid_11.0',
  'fips_6037.0',
  'heatingorsystemtypeid_18.0',
  'prod_fireplacecnt_garagecarcnt',
  'prod_numberofstories_numberofstories',
  'heatingorsystemtypeid_1.0',
  'heatingorsystemtypeid_7.0',
  'regionidcounty_2061.0',
  'airconditioningtypeid_5.0',
  'heatingorsystemtypeid_24.0',
  'heatingorsystemtypeid_13.0',
  'airconditioningtypeid_9.0',
  'threequarterbathnbr',
  'airconditioningtypeid_13.0',
  'airconditioningtypeid_1.0',
  'heatingorsystemtypeid_20.0',
  'airconditioningtypeid_11.0',
  'regionidcounty_1286.0',
  'heatingorsystemtypeid_10.0',
  'fips_6111.0',
  'heatingorsystemtypeid_2.0',
  'roomcnt',
  'calculatedfinishedsquarefeet',
  'age',
  'lotsizesquarefeet',
  'log_age',
  'prod_bedroomcnt_bedroomcnt'],
 'Gradient Boosting': ['calculatedfinishedsquarefeet',
  'buildingqualitytypeid',
  'lotsizesquarefeet',
  'roomcnt',
  'prod_fireplacecnt_garagecarcnt',
  'age',
  'regionidcounty_1286.0',
  'fips_6111.0',
  'threequarterbathnbr',
  'heatingorsystemtypeid_24.0',
  'heatingorsystemtypeid_10.0']}

In [4]:
# raw dataset csv
X_eng_dropped_train_scaled = pd.read_csv('X_eng_dropped_train_scaled.csv')
X_eng_dropped_test_scaled = pd.read_csv('X_eng_dropped_test_scaled.csv')
y_train = pd.read_csv('y_train.csv')
y_test = pd.read_csv('y_test.csv')

In [5]:
datasets = [X_eng_dropped_train_scaled, X_eng_dropped_test_scaled, y_train, y_test]
for d in datasets:
    print(d.shape)

(61656, 34)
(15414, 34)
(61656, 1)
(15414, 1)


In [6]:
y_train=np.ravel(y_train)
y_test=np.ravel(y_test)

In [6]:
# features per model 

In [3]:
def run_model(model, X_train, y_train, X_test, y_test, n_repeats=10, n_jobs=-1, **model_params):

    # Instantiate the model if a class is provided, so for example can use either BaggingRegressor or BaggingRegressor() as argument. 
    if isinstance(model, type):
        model = model(**model_params)

    neg_mse_scores = cross_val_score(model, X_train, y_train,scoring = 'neg_mean_squared_error',
                                     cv = RepeatedKFold(n_splits=5, n_repeats=n_repeats, random_state=42), n_jobs  = n_jobs)
    
    mean_cv_mse = -np.mean(neg_mse_scores)
    std_cv_mse  = np.std(neg_mse_scores)
    
    # Fit the model on the full training set
    model.fit(X_train, y_train)
    
    # Compute training MSE and testing MSE
    train_preds = model.predict(X_train)
    train_mse   = mean_squared_error(y_train, train_preds)
    test_preds  = model.predict(X_test)
    test_mse    = mean_squared_error(y_test, test_preds)
    
    return mean_cv_mse, std_cv_mse, train_mse, test_mse

In [7]:

def sweep_parameter(model,
                    Parameters,
                    param,
                    parameter_list,
                    X_train,        
                    y_train,          
                    X_test,           
                    y_test,           
                    verbose          = True,
                    show_rmse        = True,
                    n_iter_no_change = None,
                    delta            = 0.001,
                    n_jobs           = -1,
                    n_repeats        = 10):
    
    start = time.time()
    Parameters = Parameters.copy()  # Avoid modifying the original dictionary
    
    cv_mses, std_cvs, train_mses, test_mses = [], [], [], []
    no_improve_count = 0
    best_mse = float('inf')
    
    # Run over each value in parameter_list
    for p in tqdm(parameter_list, desc=f"Sweeping {param}"):
        Parameters[param] = p
        P_temp = Parameters.copy()
        # Remove MSE_found if present, just in case
        P_temp.pop('MSE_found', None)
        
        cv_mse, std_cv, train_mse, test_mse = run_model(
            model=model,
            X_train=X_train, y_train=y_train,
            X_test=X_test,   y_test=y_test,
            n_repeats=n_repeats,
            n_jobs=n_jobs,
            **P_temp
        )
        cv_mses.append(cv_mse)
        std_cvs.append(std_cv)
        train_mses.append(train_mse)
        test_mses.append(test_mse)
        
        # Early-stopping logic
        if cv_mse < best_mse - delta:
            best_mse = cv_mse
            no_improve_count = 0
        else:
            no_improve_count += 1
        
        if n_iter_no_change is not None and no_improve_count >= n_iter_no_change:
            print(f"Early stopping: No improvement after {n_iter_no_change} iterations.")
            break
    
    # Identify best parameter
    min_cv_mse = min(cv_mses)
    min_index = cv_mses.index(min_cv_mse)
    best_param = parameter_list[min_index]
    Parameters[param] = best_param
    Parameters['MSE_found'] = min_cv_mse
    
    if verbose:
        # Prepare for plotting
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 8), sharex=True)
        
        # We only need as many parameter values as we actually computed
        partial_param_list = parameter_list[:len(cv_mses)]
        
        # Check if our parameter list is Boolean so we can label accordingly
        is_boolean = all(isinstance(val, bool) for val in partial_param_list)
        if is_boolean:
            # Convert booleans to integer indices for plotting
            x_vals = list(range(len(partial_param_list)))
            x_labels = [str(val) for val in partial_param_list]
        else:
            # Treat numeric or other types as-is
            x_vals = partial_param_list
            x_labels = partial_param_list
        
        error_name = 'RMSE' if show_rmse else 'MSE'
        
        # ----- First plot: (R)MSE -----
        ax1.set_title(f"{error_name} vs {param}")
        
        # Apply dollar formatting ONLY if we're showing RMSE
        if show_rmse:
            ax1.yaxis.set_major_formatter(mticker.FuncFormatter(dollar_format))
        
        # Plot lines
        ax1.plot(x_vals,
                 np.sqrt(cv_mses) if show_rmse else cv_mses,
                 marker='.', label=f"CV {error_name}", color='blue')
        ax1.plot(x_vals,
                 np.sqrt(train_mses) if show_rmse else train_mses,
                 marker='.', label=f"Train {error_name}", color='green')
        ax1.plot(x_vals,
                 np.sqrt(test_mses) if show_rmse else test_mses,
                 linestyle='--', label=f"Test {error_name}", color='orange')
        ax1.scatter([x_vals[min_index]],
                    [np.sqrt(min_cv_mse) if show_rmse else min_cv_mse],
                    marker='x', label=f"Best CV {error_name}", color='red')
        
        ax1.set_ylabel(error_name)
        ax1.legend()
        ax1.grid()
        
        # ----- Second plot: CV Std Dev -----
        ax2.set_title(f"CV Standard Deviation vs {param}")
        ax2.plot(x_vals, std_cvs, marker='.', label=f"CV {error_name} Std", color='blue')
        ax2.set_xlabel(param)
        ax2.set_ylabel("Standard Deviation")
        ax2.legend()
        ax2.grid(alpha=0.5)
        
        # If we are using boolean x-values, set custom ticks
        if is_boolean:
            ax2.set_xticks(x_vals)
            ax2.set_xticklabels(x_labels)
        
        plt.tight_layout()
        plt.show()
        
        end = time.time()
        print("Execution Time:", time.strftime("%H:%M:%S", time.gmtime(end - start)))
    
    return Parameters

In [8]:
X_train_reduced_gb = X_eng_dropped_train_scaled[best_features_per_model['Gradient Boosting']]
X_test_reduced_gb = X_eng_dropped_test_scaled[best_features_per_model['Gradient Boosting']]

default_gradient_boosting_regressor_params = {
    "loss": "squared_error",          # Loss function to be optimized.
    "learning_rate": 0.1,             # Learning rate shrinks the contribution of each tree.
    "n_estimators": 100,              # Number of boosting stages to perform.
    "subsample": 1.0,                 # Fraction of samples to be used for fitting the individual base learners.
    "criterion": "friedman_mse",      # Criterion used for tree split quality.
    "min_samples_split": 2,           # Minimum number of samples required to split an internal node.
    "min_samples_leaf": 1,            # Minimum number of samples required to be at a leaf node.
    "min_weight_fraction_leaf": 0.0,  # Minimum weighted fraction of the sum total of weights required at a leaf node.
    "max_depth": 3,                   # Maximum depth of the individual regression estimators.
    "min_impurity_decrease": 0.0,     # A node will be split if this split induces a decrease of the impurity greater than or equal to this value.
    "init": None,                     # An estimator object that is used to compute the initial predictions.
    "random_state": None,             # Controls the random seed given to the base learners.
    "max_features": None,             # The number of features to consider when looking for the best split.
    "verbose": 0,                     # Enable verbose output.
    "max_leaf_nodes": None,           # Grow trees with max_leaf_nodes in best-first fashion.
    "warm_start": False,              # Reuse the solution of the previous call to fit and add more estimators to the ensemble.
    "validation_fraction": 0.1,        # Proportion of training data to set aside as validation set for early stopping.
    "n_iter_no_change": None,         # Used to decide if early stopping will be used to terminate training.
    "tol": 1e-4,                      # Tolerance for the early stopping. When the loss is not improving by at least tol for n_iter_no_change iterations, the training stops.
    "ccp_alpha": 0.0                  # Complexity parameter used for Minimal Cost-Complexity Pruning.
}

In [10]:
param_tests = {
    'loss': ['squared_error', 'absolute_error', 'huber', 'quantile'],
    'learning_rate': [0.001, 0.01, 0.1, 0.2, 0.3],
    'n_estimators': [50, 100, 150, 200],
    'subsample': [0.5, 0.75, 1.0],
    'criterion': ['friedman_mse', 'squared_error'],
    'min_samples_split': [2, 5, 10, 15, 20, 25],
    'min_samples_leaf': [1, 2, 4, 6, 8, 10],
    'min_weight_fraction_leaf': [0.0, 0.01, 0.1],
    'max_depth': [3, 5, 7, 9],
    'min_impurity_decrease': [0.0, 0.01, 0.1],
    'max_features': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
    'max_leaf_nodes': [None, 5, 10, 15, 20],
    'warm_start': [False, True]
}

In [10]:
param_tests_2 = {
    'max_depth': [3, 5, 7, 9, 11, 13, 15, 17, 19, 21]
}

In [11]:
# Assuming the following have already been defined or imported:
# - default_gradient_boosting_regressor_params: a dictionary with the default parameters for GradientBoostingRegressor.
# - param_tests: your dictionary of parameter lists to test.
# - X_train_reduced_gb, X_test_reduced_gb, y_train, y_test: your preprocessed datasets.
# - sweep_parameter: your function for sweeping one parameter.
# - GradientBoostingRegressor: imported from sklearn.ensemble.
# - numpy as np

# Start with the default parameters
gb_params = default_gradient_boosting_regressor_params.copy()

# Loop over each parameter in param_tests and update gb_params accordingly.
for param, values in param_tests_2.items():
    print("\nSweeping parameter:", param)
    # Run the sweep on the current parameter.
    # Here we update gb_params with the best value found for the given parameter.
    gb_params = sweep_parameter(
        model=GradientBoostingRegressor,
        Parameters=gb_params,
        param=param,
        parameter_list=values,
        X_train=X_train_reduced_gb,
        y_train=y_train,
        X_test=X_test_reduced_gb,
        y_test=y_test,
        verbose=False,       # Set verbose to False to skip charts/output if desired.
        show_rmse=True,
        n_iter_no_change=None,
        delta=0.001,
        n_jobs=-1,
        n_repeats=5
    )
    # Print the updated parameter dictionary and the best value for the current parameter.
    print("Updated parameters after sweeping", param, ":\n", gb_params)
    print("Best", param, ":", gb_params[param])
    print("Best (root) MSE:", np.sqrt(gb_params['MSE_found']))

# Optionally, print the final parameter dictionary:
print("\nFinal gradient boosting parameters:")
print(gb_params)



Sweeping parameter: max_depth


Sweeping max_depth: 100%|██████████| 10/10 [03:46<00:00, 22.65s/it]

Updated parameters after sweeping max_depth :
 {'loss': 'squared_error', 'learning_rate': 0.1, 'n_estimators': 100, 'subsample': 1.0, 'criterion': 'friedman_mse', 'min_samples_split': 2, 'min_samples_leaf': 1, 'min_weight_fraction_leaf': 0.0, 'max_depth': 5, 'min_impurity_decrease': 0.0, 'init': None, 'random_state': None, 'max_features': None, 'verbose': 0, 'max_leaf_nodes': None, 'warm_start': False, 'validation_fraction': 0.1, 'n_iter_no_change': None, 'tol': 0.0001, 'ccp_alpha': 0.0, 'MSE_found': np.float64(209243968797.1148)}
Best max_depth : 5
Best (root) MSE: 457431.9280473487

Final gradient boosting parameters:
{'loss': 'squared_error', 'learning_rate': 0.1, 'n_estimators': 100, 'subsample': 1.0, 'criterion': 'friedman_mse', 'min_samples_split': 2, 'min_samples_leaf': 1, 'min_weight_fraction_leaf': 0.0, 'max_depth': 5, 'min_impurity_decrease': 0.0, 'init': None, 'random_state': None, 'max_features': None, 'verbose': 0, 'max_leaf_nodes': None, 'warm_start': False, 'validation_f

In [12]:
# Make sure to import the required packages and models before running this block:
X_train_reduced_rf = X_eng_dropped_train_scaled[best_features_per_model['Random Forest']]
X_test_reduced_rf = X_eng_dropped_test_scaled[best_features_per_model['Random Forest']]

# Define a default parameters dictionary for RandomForestRegressor
default_random_forest_params = {
    "n_estimators": 100,           # Number of trees in the forest.
    "criterion": "squared_error",  # Loss function ("squared_error" is used in scikit-learn 1.2+).
    "max_depth": None,             # Maximum depth of the tree.
    "min_samples_split": 2,        # Minimum number of samples required to split an internal node.
    "min_samples_leaf": 1,         # Minimum number of samples required to be at a leaf node.
    "min_weight_fraction_leaf": 0.0,  # Minimum weighted fraction of the sum total of weights required at a leaf node.
    "max_features": 1.0,           # Fraction of features to consider when looking for the best split.
    "max_leaf_nodes": None,        # Grow trees with max_leaf_nodes in best-first fashion.
    "bootstrap": True,             # Whether bootstrap samples are used when building trees.
    "oob_score": False,            # Whether to use out-of-bag samples to estimate the generalization error.
    "random_state": random_state,  # Controls the randomness of the estimator.
    "verbose": 0,                  # Controls the verbosity when fitting and predicting.
    "warm_start": False            # Reuse the solution of the previous call to fit and add more estimators to the ensemble.
}


In [13]:
# Make sure to import the required packages and models before running this block:
X_train_reduced_rf = X_eng_dropped_train_scaled[best_features_per_model['Random Forest']]
X_test_reduced_rf = X_eng_dropped_test_scaled[best_features_per_model['Random Forest']]

# Define a default parameters dictionary for RandomForestRegressor
default_random_forest_params = {
    "n_estimators": 100,           # Number of trees in the forest.
    "criterion": "squared_error",  # Loss function ("squared_error" is used in scikit-learn 1.2+).
    "max_depth": None,             # Maximum depth of the tree.
    "min_samples_split": 2,        # Minimum number of samples required to split an internal node.
    "min_samples_leaf": 1,         # Minimum number of samples required to be at a leaf node.
    "min_weight_fraction_leaf": 0.0,  # Minimum weighted fraction of the sum total of weights required at a leaf node.
    "max_features": 1.0,           # Fraction of features to consider when looking for the best split.
    "max_leaf_nodes": None,        # Grow trees with max_leaf_nodes in best-first fashion.
    "bootstrap": True,             # Whether bootstrap samples are used when building trees.
    "oob_score": False,            # Whether to use out-of-bag samples to estimate the generalization error.
    "verbose": 0,                  # Controls the verbosity when fitting and predicting.
    "warm_start": False            # Reuse the solution of the previous call to fit and add more estimators to the ensemble.
}

# Define a dictionary of candidate parameter values for RandomForestRegressor.
param_tests_rf = {
    'n_estimators': [50, 100, 150, 200, 250],
    'max_depth': [None, 5, 10, 15, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': [0.5, 0.7, 1.0],
    'bootstrap': [True, False],
    'warm_start': [False, True]
}

# Start with the default parameters.
rf_params = default_random_forest_params.copy()

# Loop over each parameter in the param_tests_rf dictionary and update rf_params accordingly.
for param, values in param_tests_rf.items():
    print("\nSweeping parameter:", param)
    # Sweep the current parameter over its candidate values.
    rf_params = sweep_parameter(
        model=RandomForestRegressor,
        Parameters=rf_params,
        param=param,
        parameter_list=values,
        X_train=X_train_reduced_rf,  # Preprocessed training data for RandomForest
        y_train=y_train,             # Training targets
        X_test=X_test_reduced_rf,    # Preprocessed test data for RandomForest
        y_test=y_test,               # Test targets
        verbose=False,               # Set to False to skip plotting charts during the sweep.
        show_rmse=True,
        n_iter_no_change=None,
        delta=0.001,
        n_jobs=-1,
        n_repeats=5
    )
    # Print updated parameters and best value for the current parameter sweep.
    print("Updated parameters after sweeping", param, ":\n", rf_params)
    print("Best", param, ":", rf_params[param])
    print("Best (root) MSE:", np.sqrt(rf_params['MSE_found']))

# Print final RandomForestRegressor parameters after all sweeps.
print("\nFinal RandomForestRegressor parameters:")
print(rf_params)



Sweeping parameter: n_estimators


Sweeping n_estimators: 100%|██████████| 5/5 [05:22<00:00, 64.50s/it]


Updated parameters after sweeping n_estimators :
 {'n_estimators': 250, 'criterion': 'squared_error', 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 1, 'min_weight_fraction_leaf': 0.0, 'max_features': 1.0, 'max_leaf_nodes': None, 'bootstrap': True, 'oob_score': False, 'verbose': 0, 'warm_start': False, 'MSE_found': np.float64(210659903381.29648)}
Best n_estimators : 250
Best (root) MSE: 458977.018358541

Sweeping parameter: max_depth


Sweeping max_depth: 100%|██████████| 5/5 [05:25<00:00, 65.14s/it] 


Updated parameters after sweeping max_depth :
 {'n_estimators': 250, 'criterion': 'squared_error', 'max_depth': 15, 'min_samples_split': 2, 'min_samples_leaf': 1, 'min_weight_fraction_leaf': 0.0, 'max_features': 1.0, 'max_leaf_nodes': None, 'bootstrap': True, 'oob_score': False, 'verbose': 0, 'warm_start': False, 'MSE_found': np.float64(208187704870.25778)}
Best max_depth : 15
Best (root) MSE: 456275.908711229

Sweeping parameter: min_samples_split


...

Updated parameters after sweeping max_features :
 {'n_estimators': 250, 'criterion': 'squared_error', 'max_depth': 15, 'min_samples_split': 10, 'min_samples_leaf': 4, 'min_weight_fraction_leaf': 0.0, 'max_features': 0.5, 'max_leaf_nodes': None, 'bootstrap': True, 'oob_score': False, 'verbose': 0, 'warm_start': False, 'MSE_found': np.float64(196985657084.0791)}
Best max_features : 0.5
Best (root) MSE: 443830.6626226709

Sweeping parameter: bootstrap


Sweeping bootstrap: 100%|██████████| 2/2 [01:22<00:00, 41.16s/it]


Updated parameters after sweeping bootstrap :
 {'n_estimators': 250, 'criterion': 'squared_error', 'max_depth': 15, 'min_samples_split': 10, 'min_samples_leaf': 4, 'min_weight_fraction_leaf': 0.0, 'max_features': 0.5, 'max_leaf_nodes': None, 'bootstrap': True, 'oob_score': False, 'verbose': 0, 'warm_start': False, 'MSE_found': np.float64(197203284114.00238)}
Best bootstrap : True
Best (root) MSE: 444075.76393449167

Sweeping parameter: warm_start


Sweeping warm_start: 100%|██████████| 2/2 [01:06<00:00, 33.03s/it]

Updated parameters after sweeping warm_start :
 {'n_estimators': 250, 'criterion': 'squared_error', 'max_depth': 15, 'min_samples_split': 10, 'min_samples_leaf': 4, 'min_weight_fraction_leaf': 0.0, 'max_features': 0.5, 'max_leaf_nodes': None, 'bootstrap': True, 'oob_score': False, 'verbose': 0, 'warm_start': False, 'MSE_found': np.float64(197066374456.80887)}
Best warm_start : False
Best (root) MSE: 443921.58593248075

Final RandomForestRegressor parameters:
{'n_estimators': 250, 'criterion': 'squared_error', 'max_depth': 15, 'min_samples_split': 10, 'min_samples_leaf': 4, 'min_weight_fraction_leaf': 0.0, 'max_features': 0.5, 'max_leaf_nodes': None, 'bootstrap': True, 'oob_score': False, 'verbose': 0, 'warm_start': False, 'MSE_found': np.float64(197066374456.80887)}


In [14]:
# Make sure to import the required packages and models before running this block:
X_train_reduced_ridge = X_eng_dropped_train_scaled[best_features_per_model['Ridge Regression']]
X_test_reduced_ridge = X_eng_dropped_test_scaled[best_features_per_model['Ridge Regression']]

# Define default parameters for Ridge Regression.
default_ridge_params = {
    "alpha": 1.0,            # Regularization strength.
    "fit_intercept": True,   # Whether to calculate the intercept.
    "copy_X": True,          # Whether to copy X (if False, it may overwrite X).
    "max_iter": None,        # Maximum number of iterations (None means no limit).
    "tol": 0.001,            # Tolerance for convergence.
    "solver": "auto",        # Solver to use in the computational routines.
    "random_state": random_state  # Assuming random_state is defined in your environment.
}

# Define a dictionary of candidate parameter values to sweep for Ridge Regression.
param_tests_ridge = {
    "alpha": [0.01, 0.1, 1.0, 10.0, 100.0],
    "tol": [0.0001, 0.001, 0.01],
    "solver": ["auto", "svd", "cholesky"]
}

# Assume that these have been defined in your environment:
# X_train_reduced_ridge: the training data subset for Ridge.
# X_test_reduced_ridge: the test data subset for Ridge.
# y_train, y_test: the target arrays.

# Start with the default parameters.
ridge_params = default_ridge_params.copy()

# Loop over each parameter defined in param_tests_ridge and update ridge_params.
for param, values in param_tests_ridge.items():
    print("\nSweeping parameter:", param)
    
    ridge_params = sweep_parameter(
        model=Ridge,
        Parameters=ridge_params,
        param=param,
        parameter_list=values,
        X_train=X_train_reduced_ridge,  # Preprocessed training data for Ridge Regression.
        y_train=y_train,                # Training targets.
        X_test=X_test_reduced_ridge,    # Preprocessed test data for Ridge Regression.
        y_test=y_test,                  # Test targets.
        verbose=False,                  # Skip chart plotting.
        show_rmse=True,
        n_iter_no_change=None,
        delta=0.001,
        n_jobs=-1,
        n_repeats=5
    )
    
    # Print out the updated parameter dictionary after each sweep.
    print("Updated parameters after sweeping", param, ":\n", ridge_params)
    print("Best", param, ":", ridge_params[param])
    print("Best (root) MSE:", np.sqrt(ridge_params['MSE_found']))

# Print the final Ridge Regression parameters after all sweeps.
print("\nFinal Ridge Regression parameters:")
print(ridge_params)



Sweeping parameter: alpha


Sweeping alpha: 100%|██████████| 5/5 [00:00<00:00,  5.57it/s]


Updated parameters after sweeping alpha :
 {'alpha': 100.0, 'fit_intercept': True, 'copy_X': True, 'max_iter': None, 'tol': 0.001, 'solver': 'auto', 'random_state': 42, 'MSE_found': np.float64(233047438239.80316)}
Best alpha : 100.0
Best (root) MSE: 482749.8712996236

Sweeping parameter: tol


Sweeping tol: 100%|██████████| 3/3 [00:00<00:00,  5.89it/s]


Updated parameters after sweeping tol :
 {'alpha': 100.0, 'fit_intercept': True, 'copy_X': True, 'max_iter': None, 'tol': 0.0001, 'solver': 'auto', 'random_state': 42, 'MSE_found': np.float64(233047438239.80316)}
Best tol : 0.0001
Best (root) MSE: 482749.8712996236

Sweeping parameter: solver


Sweeping solver: 100%|██████████| 3/3 [00:00<00:00,  5.07it/s]

Updated parameters after sweeping solver :
 {'alpha': 100.0, 'fit_intercept': True, 'copy_X': True, 'max_iter': None, 'tol': 0.0001, 'solver': 'auto', 'random_state': 42, 'MSE_found': np.float64(233047438239.80316)}
Best solver : auto
Best (root) MSE: 482749.8712996236

Final Ridge Regression parameters:
{'alpha': 100.0, 'fit_intercept': True, 'copy_X': True, 'max_iter': None, 'tol': 0.0001, 'solver': 'auto', 'random_state': 42, 'MSE_found': np.float64(233047438239.80316)}


In [ ]:
import time
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import RandomizedSearchCV, RepeatedKFold

# Assume X_train and y_train are already defined (e.g., after any preprocessing)
# Here is your parameter grid for GradientBoostingRegressor:
param_tests = {
    'loss': ['squared_error', 'absolute_error', 'huber', 'quantile'],
    'learning_rate': [0.001, 0.01, 0.1, 0.2, 0.3],
    'n_estimators': [50, 100, 150, 200],
    'subsample': [0.5, 0.75, 1.0],
    'criterion': ['friedman_mse', 'squared_error'],
    'min_samples_split': [2, 5, 10, 15, 20, 25],
    'min_samples_leaf': [1, 2, 4, 6, 8, 10],
    'min_weight_fraction_leaf': [0.0, 0.01, 0.1],
    'max_depth': [3, 5, 7, 9],
    'min_impurity_decrease': [0.0, 0.01, 0.1],
    'max_features': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
    'max_leaf_nodes': [None, 5, 10, 15, 20],
    'warm_start': [False, True]
}

# Record the start time
start_time = time.time()

# Instantiate the GradientBoostingRegressor with a fixed random state for reproducibility.
gb_model = GradientBoostingRegressor(random_state=42)

# Set up a cross-validator; here we use 3-fold repeated 3 times.
cv = RepeatedKFold(n_splits=5, n_repeats=3, random_state=42)

# Set up the RandomizedSearchCV.
# n_iter controls how many random parameter combinations will be evaluated.
random_search = RandomizedSearchCV(
    estimator=gb_model,
    param_distributions=param_tests,
    n_iter=100,                          # Adjust this for more/less thorough search
    scoring='neg_mean_squared_error',   # Negative MSE (will be converted later)
    cv=cv,
    n_jobs=-1,
    verbose=1,                          # Progress messages printed to console
    random_state=42
)

# Fit the model on the training data
random_search.fit(X_train_reduced_gb, y_train)

# Extract the results as a DataFrame.
results_df = pd.DataFrame(random_search.cv_results_)

# Convert negative MSE back to positive MSE and then compute RMSE.
results_df['mean_test_MSE'] = -results_df['mean_test_score']
results_df['mean_test_RMSE'] = np.sqrt(results_df['mean_test_MSE'])

# Sort the results by RMSE (lowest is best).
sorted_results = results_df.sort_values(by='mean_test_RMSE')

# Display the top 10 parameter combinations by RMSE.
print("\nTop 10 Parameter Combinations by RMSE:")
print(sorted_results[['param_loss', 'param_learning_rate', 'param_n_estimators',
                        'param_subsample', 'param_criterion', 'param_min_samples_split',
                        'param_min_samples_leaf', 'param_min_weight_fraction_leaf',
                        'param_max_depth', 'param_min_impurity_decrease',
                        'param_max_features', 'param_max_leaf_nodes', 'param_warm_start',
                        'mean_test_RMSE']].head(10))

# Print the best parameters and corresponding RMSE.
best_params_gb = random_search.best_params_
best_rmse_gb = np.sqrt(-random_search.best_score_)

print(f"\nBest Parameters: {best_params_gb}")
print(f"Best RMSE: {best_rmse_gb:.4f}")

# Show execution time.
execution_time = time.time() - start_time
print(f"\nExecution Time: {execution_time:.2f}s")


Fitting 15 folds for each of 100 candidates, totalling 1500 fits

Top 10 Parameter Combinations by RMSE:
        param_loss  param_learning_rate  param_n_estimators  param_subsample  \
15   squared_error                 0.10                  50             0.75   
73           huber                 0.20                 150             0.75   
77   squared_error                 0.10                 150             0.75   
83  absolute_error                 0.30                 100             0.50   
9    squared_error                 0.30                 100             1.00   
10           huber                 0.30                 200             1.00   
87  absolute_error                 0.20                 150             0.50   
69           huber                 0.20                 100             1.00   
39  absolute_error                 0.20                  50             0.75   
81   squared_error                 0.01                 200             0.50   

   param_crite

In [16]:
# Parameter grid for RandomForestRegressor tuning.
param_tests_rf = {
    'n_estimators': [50, 100, 150, 200, 250],
    'max_depth': [None, 5, 10, 15, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': [0.5, 0.7, 1.0],
    'bootstrap': [True, False],
    'warm_start': [False, True]
}

# Record the start time.
start_time = time.time()

# Instantiate the RandomForestRegressor with a fixed random state.
rf_model = RandomForestRegressor(random_state=42)

# Set up cross-validation: here we use 5-fold cross-validation repeated 3 times.
cv = RepeatedKFold(n_splits=5, n_repeats=3, random_state=42)

# Set up the RandomizedSearchCV.
# n_iter controls the number of random combinations to try.
random_search_rf = RandomizedSearchCV(
    estimator=rf_model,
    param_distributions=param_tests_rf,
    n_iter=50,                          # Adjust n_iter for more/less thorough search.
    scoring='neg_mean_squared_error',   # We use negative MSE; will convert later.
    cv=cv,
    n_jobs=-1,
    verbose=1,                          # Verbose prints progress.
    random_state=42
)

# Fit the model using the reduced training data for RandomForest.
random_search_rf.fit(X_train_reduced_rf, y_train)

# Extract the results into a DataFrame.
results_rf_df = pd.DataFrame(random_search_rf.cv_results_)

# Convert negative MSE back to positive MSE and then compute RMSE.
results_rf_df['mean_test_MSE'] = -results_rf_df['mean_test_score']
results_rf_df['mean_test_RMSE'] = np.sqrt(results_rf_df['mean_test_MSE'])

# Sort the results by RMSE (lowest is best).
sorted_results_rf = results_rf_df.sort_values(by='mean_test_RMSE')

# Display the top 10 parameter combinations (by RMSE).
print("\nTop 10 Parameter Combinations by RMSE:")
print(sorted_results_rf[['param_n_estimators', 
                           'param_max_depth', 
                           'param_min_samples_split',
                           'param_min_samples_leaf', 
                           'param_max_features', 
                           'param_bootstrap',
                           'param_warm_start',
                           'mean_test_RMSE']].head(10))

# Get and print the best parameters and corresponding RMSE.
best_params_rf = random_search_rf.best_params_
best_rmse_rf = np.sqrt(-random_search_rf.best_score_)

print(f"\nBest Parameters: {best_params_rf}")
print(f"Best RMSE: {best_rmse_rf:.4f}")

# Display the execution time.
execution_time_rf = time.time() - start_time
print(f"\nExecution Time: {execution_time_rf:.2f}s")


Fitting 15 folds for each of 50 candidates, totalling 750 fits

Top 10 Parameter Combinations by RMSE:
    param_n_estimators param_max_depth  param_min_samples_split  \
32                 250              15                        2   
19                 250              15                       10   
2                  200              20                       10   
6                  150              15                        2   
16                  50              15                        5   
15                 100              20                        5   
1                  100              20                       10   
3                  100              15                        2   
4                   50              20                        2   
18                 100              15                        5   

    param_min_samples_leaf  param_max_features  param_bootstrap  \
32                       4                 0.5             True   
19                       

In [17]:
import time
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.model_selection import RandomizedSearchCV, RepeatedKFold

# Define the parameter grid/distribution for Ridge Regression.
param_tests_ridge = {
    "alpha": [0.01, 0.1, 1.0, 10.0, 100.0],
    "tol": [0.0001, 0.001, 0.01],
    "solver": ["auto", "svd", "cholesky"]
}

# Record the start time.
start_time = time.time()

# Instantiate the Ridge regressor.
ridge_model = Ridge()

# Set up a cross-validator. Here we use 5-fold CV repeated 3 times.
cv = RepeatedKFold(n_splits=5, n_repeats=3, random_state=42)

# Set up the RandomizedSearchCV for Ridge.
# n_iter controls the number of random parameter combinations evaluated.
random_search_ridge = RandomizedSearchCV(
    estimator=ridge_model,
    param_distributions=param_tests_ridge,
    n_iter=20,                          # Adjust n_iter to balance thorough search vs. speed.
    scoring='neg_mean_squared_error',   # We use neg MSE; later we'll convert to RMSE.
    cv=cv,
    n_jobs=-1,
    verbose=1,
    random_state=42
)

# Fit the model using the training data.
random_search_ridge.fit(X_train_reduced_ridge, y_train)

# Extract the CV results into a DataFrame.
results_ridge_df = pd.DataFrame(random_search_ridge.cv_results_)

# Convert negative MSE back to positive and then compute RMSE.
results_ridge_df['mean_test_MSE'] = -results_ridge_df['mean_test_score']
results_ridge_df['mean_test_RMSE'] = np.sqrt(results_ridge_df['mean_test_MSE'])

# Sort the results by RMSE, where lower values indicate better performance.
sorted_results_ridge = results_ridge_df.sort_values(by='mean_test_RMSE')

# Display the top 10 parameter combinations along with their RMSE.
print("\nTop 10 Parameter Combinations by RMSE:")
print(sorted_results_ridge[['param_alpha', 'param_tol', 'param_solver', 'mean_test_RMSE']].head(10))

# Get and print the best parameters and corresponding RMSE.
best_params_ridge = random_search_ridge.best_params_
best_rmse_ridge = np.sqrt(-random_search_ridge.best_score_)

print(f"\nBest Parameters: {best_params_ridge}")
print(f"Best RMSE: {best_rmse_ridge:.4f}")

# Display the total execution time.
execution_time_ridge = time.time() - start_time
print(f"\nExecution Time: {execution_time_ridge:.2f}s")


Fitting 15 folds for each of 20 candidates, totalling 300 fits

Top 10 Parameter Combinations by RMSE:
    param_alpha  param_tol param_solver  mean_test_RMSE
3         100.0     0.0010     cholesky   482684.251648
16        100.0     0.0001     cholesky   482684.251648
5         100.0     0.0100          svd   482684.251648
0         100.0     0.0001          svd   482684.251648
4          10.0     0.0100     cholesky   482687.670140
12         10.0     0.0100          svd   482687.670140
11          1.0     0.0001     cholesky   482688.575660
2           1.0     0.0100     cholesky   482688.575660
13          1.0     0.0010         auto   482688.575660
1           1.0     0.0010     cholesky   482688.575660

Best Parameters: {'tol': 0.001, 'solver': 'cholesky', 'alpha': 100.0}
Best RMSE: 482684.2516

Execution Time: 1.04s


### GridSearchCV

In [ ]:
TOP_N = 10  # e.g., top 10 combos
top_n_results = sorted_results.head(TOP_N)

# Build a new narrower parameter grid
def extract_params_from_top_n(df, param_name):
    """
    Gather all distinct parameter values for 'param_name'
    from the top N results.
    """
    return df[param_name].unique().tolist()

# Identify which params we want to refine. For Ridge:
params_of_interest = ['loss', 'learning_rate', 'n_estimators', 'subsample']

# Create a new dictionary for the refined param grid
refined_param_grid = {}
for p in params_of_interest:
    col_name = f"param_{p}"
    refined_param_grid[p] = extract_params_from_top_n(top_n_results, col_name)

print("Refined parameter grid based on top N from random search:")
print(refined_param_grid)

# Perform a GridSearchCV on this refined grid

# Set up our cross-validator:
cv = RepeatedKFold(n_splits=5, n_repeats=3, random_state=42)

# Create a Ridge model (optionally setting random_state if desired).
gb_model = GradientBoostingRegressor()

grid_search_refined = GridSearchCV(
    estimator=gb_model,
    param_grid=refined_param_grid,
    scoring='neg_mean_squared_error',  # or another appropriate metric
    cv=cv,
    n_jobs=-1,
    verbose=1
)

# We'll assume X_train_reduced_ridge and y_train hold our training data.
grid_search_refined.fit(X_train_reduced_ridge, y_train)

# Convert negative MSE back to RMSE and output the results.
best_params_refined = grid_search_refined.best_params_
best_rmse_refined = np.sqrt(-grid_search_refined.best_score_)

print("\nRefined GridSearchCV Results:")
print(f"Best Parameters: {best_params_refined}")
print(f"Best RMSE: {best_rmse_refined:.4f}")


Refined parameter grid based on top N from random search:
{'loss': ['squared_error', 'huber', 'absolute_error'], 'learning_rate': [0.1, 0.2, 0.3, 0.01], 'n_estimators': [50, 150, 100, 200], 'subsample': [0.75, 0.5, 1.0]}
Fitting 15 folds for each of 144 candidates, totalling 2160 fits

Refined GridSearchCV Results:
Best Parameters: {'learning_rate': 0.1, 'loss': 'squared_error', 'n_estimators': 150, 'subsample': 1.0}
Best RMSE: 459276.6178


In [ ]:
TOP_N = 10  # e.g., top 10 combos
top_n_results = sorted_results_ridge.head(TOP_N)

# identify which params we want to refine
params_of_interest = ["alpha", "tol", "solver"]

# Create a new dictionary for the refined param grid
refined_param_grid = {}
for p in params_of_interest:
    col_name = f"param_{p}"
    refined_param_grid[p] = extract_params_from_top_n(top_n_results, col_name)

print("Refined parameter grid based on top N from random search:")
print(refined_param_grid)

# Perform a GridSearchCV on this refined grid

# We'll set up our cross-validator:
cv = RepeatedKFold(n_splits=5, n_repeats=3, random_state=42)

# Create a Ridge model (optionally setting random_state if desired).
ridge_model = Ridge()

grid_search_refined = GridSearchCV(
    estimator=ridge_model,
    param_grid=refined_param_grid,
    scoring='neg_mean_squared_error',  # or another appropriate metric
    cv=cv,
    n_jobs=-1,
    verbose=1
)

# We'll assume X_train_reduced_ridge and y_train hold our training data.
grid_search_refined.fit(X_train_reduced_ridge, y_train)

# Convert negative MSE back to RMSE and output the results.
best_params_refined = grid_search_refined.best_params_
best_rmse_refined = np.sqrt(-grid_search_refined.best_score_)

print("\nRefined GridSearchCV Results:")
print(f"Best Parameters: {best_params_refined}")
print(f"Best RMSE: {best_rmse_refined:.4f}")


Refined parameter grid based on top N from random search:
{'alpha': [100.0, 10.0, 1.0], 'tol': [0.001, 0.0001, 0.01], 'solver': ['cholesky', 'svd', 'auto']}
Fitting 15 folds for each of 27 candidates, totalling 405 fits

Refined GridSearchCV Results:
Best Parameters: {'alpha': 100.0, 'solver': 'cholesky', 'tol': 0.001}
Best RMSE: 482684.2516


looks like we've got some random noise so perhaps the gridsearchcv isn't that necessary given the extent of our sweeps 

### final model runs

In [ ]:
#set final parameters and test/train sets for each model
final_params = {
    'Gradient Boosting': best_params_gb,
    'Random Forest': best_params_rf,
    'Ridge Regression': best_params_ridge
}

test_sets = {
    'Gradient Boosting': (X_test_reduced_gb, y_test),
    'Random Forest': (X_test_reduced_rf, y_test),
    'Ridge Regression': (X_test_reduced_ridge, y_test)
}

train_sets = {
    'Gradient Boosting': (X_train_reduced_gb, y_train),
    'Random Forest': (X_train_reduced_rf, y_train),
    'Ridge Regression': (X_train_reduced_ridge, y_train)
}

# Prepare a list to hold each model's results.
final_model_runs = []

# Loop over each model in final_params, using tqdm to show progress.
for model_name, params in tqdm(final_params.items(), desc="Running Final Models", total=len(final_params)):
    print(f"\nRunning {model_name} with best parameters:")
    print(params)
    
    # Unpack the training and test sets for the current model.
    X_train_set, y_train_set = train_sets[model_name]
    X_test_set, y_test_set = test_sets[model_name]
    
    # Select the appropriate model class.
    if model_name == 'Gradient Boosting':
        model_class = GradientBoostingRegressor
    elif model_name == 'Random Forest':
        model_class = RandomForestRegressor
    elif model_name == 'Ridge Regression':
        model_class = Ridge
    else:
        raise ValueError(f"Unknown model: {model_name}")
        
    # Run the model with our best parameters.
    cv_mse, cv_std, train_mse, test_mse = run_model(
        model=model_class,
        X_train=X_train_set,
        y_train=y_train_set,
        X_test=X_test_set,
        y_test=y_test_set,
        n_repeats=5,
        n_jobs=-1,
        random_state=random_state,
        **params
    )
    
    # Compute RMSE values from MSE.
    cv_rmse = np.sqrt(cv_mse)
    train_rmse = np.sqrt(train_mse)
    test_rmse = np.sqrt(test_mse)
    
    # Create a dictionary of the results.
    result_dict = {
        "Model": model_name,
        "Parameters": params,
        "CV_MSE": cv_mse,
        "CV_RMSE": cv_rmse,
        "CV_Std": cv_std,
        "Train_MSE": train_mse,
        "Train_RMSE": train_rmse,
        "Test_MSE": test_mse,
        "Test_RMSE": test_rmse
    }
    
    # Append the dictionary to our result list.
    final_model_runs.append(result_dict)
    
    # Print the individual model results.
    print("Final Model Results:")
    print("CV MSE:", cv_mse)
    print("CV RMSE:", cv_rmse)
    print("CV Std Dev:", cv_std)
    print("Train MSE:", train_mse)
    print("Train RMSE:", train_rmse)
    print("Test MSE:", test_mse)
    print("Test RMSE:", test_rmse)

# Convert the list of results into a DataFrame.
final_model_runs = pd.DataFrame(final_model_runs)

# Display the final results DataFrame.
print("\nFinal Model Runs:")
print(final_model_runs[['Model', 'Train_RMSE', 'Test_RMSE']].sort_values(by='Test_RMSE'))

Running Final Models:   0%|          | 0/3 [00:00<?, ?it/s]


Running Gradient Boosting with best parameters:
{'warm_start': False, 'subsample': 0.75, 'n_estimators': 50, 'min_weight_fraction_leaf': 0.0, 'min_samples_split': 2, 'min_samples_leaf': 8, 'min_impurity_decrease': 0.1, 'max_leaf_nodes': 15, 'max_features': 0.4, 'max_depth': 5, 'loss': 'squared_error', 'learning_rate': 0.1, 'criterion': 'squared_error'}


Running Final Models:  33%|███▎      | 1/3 [00:01<00:03,  1.78s/it]

Final Model Results:
CV MSE: 206825160745.62405
CV RMSE: 454780.34340286086
CV Std Dev: 25864513355.028248
Train MSE: 179000813650.99222
Train RMSE: 423084.8775966735
Test MSE: 303538528381.2382
Test RMSE: 550943.3077742556

Running Random Forest with best parameters:
{'warm_start': True, 'n_estimators': 250, 'min_samples_split': 2, 'min_samples_leaf': 4, 'max_features': 0.5, 'max_depth': 15, 'bootstrap': True}


Running Final Models: 100%|██████████| 3/3 [00:34<00:00, 11.59s/it]

Final Model Results:
CV MSE: 197222596200.4506
CV RMSE: 444097.5075368591
CV Std Dev: 25550563046.570885
Train MSE: 124641592326.88951
Train RMSE: 353046.1617506831
Test MSE: 297721999083.3628
Test RMSE: 545639.0740071341

Running Ridge Regression with best parameters:
{'tol': 0.001, 'solver': 'cholesky', 'alpha': 100.0}
Final Model Results:
CV MSE: 233047438239.80316
CV RMSE: 482749.8712996236
CV Std Dev: 29889791460.504013
Train MSE: 232235221483.5713
Train RMSE: 481907.897303594
Test MSE: 316988555696.16473
Test RMSE: 563017.3671354773

Final Model Runs:
               Model     Train_RMSE      Test_RMSE
1      Random Forest  353046.161751  545639.074007
0  Gradient Boosting  423084.877597  550943.307774
2   Ridge Regression  481907.897304  563017.367135
